# [EDA] 2022년도 전체 시도 세부사업 예산현황 분리·정제 (작업중) — 이슈 #8

> **이 노트북은 2024년(#5) 노트북을 템플릿으로 파생한 2022년 작업본이다.**
> - 코드(로드/분류/분리/QA 계산)는 연도 파라미터만 2022/2021로 맞춰 그대로 재사용.
> - 입력: `data/raw/*2022*.xlsx` (단일 파일, `정리본_자동`·`Table 1` 시트 포함).
> - LLM 체크포인트는 `2022_llm_정제_체크포인트.csv`로 분리(2024 것과 안 섞임).
> - **아래 마크다운 서술과 QA 진단(충남 등)은 2024년 결과다. 2022년 실행 후 실제 결과로 교체할 것.**

In [ ]:
# 라이브러리 임포트
import re
from pathlib import Path

import pandas as pd
import numpy as np

# 경로 설정
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORTS_DIR = Path("../reports")

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

In [ ]:
# 데이터 로드 (2022: 단일 xlsx에 Table 1 / 정리본_자동 / 검증_자동 시트 모두 포함)
file_2022 = next(DATA_DIR.glob("*2022*.xlsx"))
df_raw = pd.read_excel(file_2022, sheet_name="정리본_자동")

print(df_raw.shape)
df_raw.head(10)

In [ ]:
# 기본검사
print("[데이터셋 크기]\n", df_raw.shape)
print("=====================================")
print("[데이터 타입]\n", df_raw.dtypes)
print("=====================================")
print("[각 컬럼별 결측치 개수 평균]\n", df_raw.isna().mean().round(3))
print("=====================================")
print("[중복 행 수]", df_raw.duplicated().sum())
print("=====================================")

# 지역(시도) 목록 확인
print("[지역(시도) 목록 확인]\n", df_raw["지역"].value_counts())

In [ ]:
# 지역별로 데이터 분리
sido_dfs = {sido: group.copy() for sido, group in df_raw.groupby("지역")}

# 시도별 행 수 확인
for sido, df_sido in sido_dfs.items():
    print(sido, len(df_sido))

In [ ]:
# 서울만 따로 확인 - #3 산출물 검증
df_seoul = sido_dfs["서울"]

print(df_seoul.shape)
df_seoul.head(10)

In [ ]:
# 이전 데이터셋에서 확인한 classify_row 적용
sido_title_pattern = re.compile(r"붙\s*임\s*\(([^)]+)\)")
# 2022 페이지 반복헤더/시도경계 제목행 (2024 붙임 패턴으로는 안 걸리는 형식)
header_noise_pattern = re.compile(
    r"고령사회기본계획|지방자치단체\s*시행계획|세부사업별\s*예산\s*현황"
)


def classify_row(세부사업명: str) -> str:
    """대분류_소계 / 중분류_소계 / 헤더반복 / 세부사업 행 판별"""
    if pd.isna(세부사업명) or str(세부사업명).strip() == "":
        return "헤더반복"

    name = str(세부사업명).strip()

    if name == "세부사업명":
        return "헤더반복"
    if sido_title_pattern.search(name):
        return "헤더반복"
    if header_noise_pattern.search(name):
        return "헤더반복"
    if re.match(
        r"^[Ⅰ-Ⅿ]", name
    ):  # 유니코드 로마숫자 대문자 전체 블록(Ⅰ~ⅬⅭⅮⅯ) - 대분류 10개(Ⅹ) 초과 대비
        return "대분류_소계"
    if re.match(r"^\d+\.", name) and re.search(r"\((공통|자체)\)$", name):  # 조건 추가
        return "중분류_소계"
    return "세부사업"


df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)
df_raw["2022년 예산"] = pd.to_numeric(
    df_raw["2022년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
)

-> #3의 결과와 정확하게 일치하는 것 확인

-> 이 노트북에서 서울은 따로 데이터셋 정제 불필요

In [ ]:
df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 시도별 행종류 분포
rowtype_distribution = df_raw.groupby(["지역", "사업행구분"]).size().unstack(fill_value=0)
rowtype_distribution

-> 서울은 0건, 나머지 16개 시도는 시도경계 제목행("붙임 (OO시 OO교육청)") 1건씩 확인

-> 정리본_자동 sheet에서 정제된 것 확인 

-> 대분류_소계도 2개로 일관적임을 확인

-> 경기 경남에서 중분류_소계가 9건이라 확인 필요

In [ ]:
# 경기 중분류_소계 행 확인
df_gyeonggi = sido_dfs["경기"]
df_gyeonggi["사업행구분"] = df_gyeonggi["세부사업명"].apply(classify_row)
print("[경기] 중분류_소계 행 확인")
print("==================================================")
print(df_gyeonggi[df_gyeonggi["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

print("\n")

# 경남도 동일하게 확인
df_gyeongnam = sido_dfs["경남"]
df_gyeongnam["사업행구분"] = df_gyeongnam["세부사업명"].apply(classify_row)
print("[경남] 중분류_소계 행 확인")
print("==================================================")
print(df_gyeongnam[df_gyeongnam["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

-> 진짜 중분류_소계 행들을 보면 뒤에 (공통) / (자체) 가 오는 것을 확인할 수 있음. 

-> 이  끝부분 조건 추가하면 오분류 걸러내기 가능함. 

-> classify_row 함수 수정 

-> 조건 추가하니까 해결됨

In [ ]:
# 지역별로 원본 순서 유지한 채 대분류/중분류 라벨 세부사업행에 전파
def assign_labels(df_sido: pd.DataFrame) -> pd.DataFrame:
    """대분류_소계 / 중분류_소계 행의 이름을 뒤따르는 세부사업 행에 전파"""
    df_sido = df_sido.sort_values(
        "원본행"
    ).copy()  # 그냥 copy해도 되지만(raw도 이미 순서가 정렬되어있음) ffill이 순서에 의존하므로 원본 문서 순서를 명시적으로 보장함.
    major_mask = df_sido["사업행구분"] == "대분류_소계"
    medium_mask = df_sido["사업행구분"] == "중분류_소계"
    df_sido["대분류"] = df_sido["세부사업명"].where(major_mask).ffill()
    df_sido["중분류"] = df_sido["세부사업명"].where(medium_mask).ffill()

    return df_sido


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# leaf(세부사업)만 추출
df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_leaf.shape)
print(df_leaf.columns.tolist())
df_leaf.head()

In [ ]:
mask_non_numeric = (
    pd.to_numeric(df_leaf["2022년 예산"], errors="coerce").isna() & df_leaf["2022년 예산"].notna()
)
display(df_leaf.loc[mask_non_numeric, ["지역", "세부사업명", "2022년 예산"]])

In [ ]:
# QA: 중분류_소계 행 예산 vs leaf 합계 대조 (17개 시도 x 중분류 단위)
leaf_sum = (
    df_leaf.groupby(["지역", "중분류"])["2022년 예산"]
    .sum()
    .reset_index()
    .rename(columns={"2022년 예산": "leaf_합계"})
)

# 원본 소계 행 값 (중분류_소계 행에서 직접 가져온 값)
subtotal = df_labeled[df_labeled["사업행구분"] == "중분류_소계"][
    ["지역", "대분류", "세부사업명", "2022년 예산"]
].rename(columns={"세부사업명": "중분류", "2022년 예산": "원본_소계값"})

qa = subtotal.merge(leaf_sum, on=["지역", "중분류"], how="left")
qa["결과"] = np.where(qa["leaf_합계"] == qa["원본_소계값"], "일치", "불일치")
qa["결과"].value_counts()

In [ ]:
qa.head()

In [ ]:
qa["차이"] = qa["leaf_합계"] - qa["원본_소계값"]

# 결과 컬럼을 항상 채워서, 저장 파일이 비어있어도 "검증은 했다"는 게 남도록 함
qa["결과"] = qa["차이"].abs().le(0.01).map({True: "일치", False: "불일치"})

qa_mismatch = qa[qa["결과"] == "불일치"]
print(f"검증 대상: {len(qa)}건 / 불일치: {len(qa_mismatch)}건")

In [ ]:
display(qa_mismatch[["지역", "대분류", "중분류", "원본_소계값", "leaf_합계", "차이"]])

# 충남 QA 불일치 원인 확인 - 원본 Table 1 대조

`정리본_자동`에서 충남만 QA 불일치가 크게 나온 원인이 `정리본_자동` 변환 과정의 데이터 손실인지, 원본 문서 자체의 문제인지 확인한다.

`원본행` 컬럼은 `Table 1` 시트의 1-indexed 엑셀 행 번호를 그대로 가리키므로, 이 값으로 원본 위치를 바로 대조할 수 있다.

In [ ]:
df_chungnam_raw = sido_dfs["충남"]
mask_nan = pd.to_numeric(
    df_chungnam_raw["2022년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
).isna()

display(df_chungnam_raw.loc[mask_nan, ["세부사업명", "2022년 예산"]])

In [ ]:
df_chungnam = df_labeled[df_labeled["지역"] == "충남"]
print(df_chungnam["사업행구분"].value_counts())
display(df_chungnam[df_chungnam["사업행구분"] == "중분류_소계"]["세부사업명"])

In [ ]:
df_chungnam_check = df_labeled[df_labeled["지역"] == "충남"][
    ["원본행", "세부사업명", "사업행구분", "대분류", "중분류"]
]

display(df_chungnam_check.head(30))

In [ ]:
# 원본 Table 1 로드 (QA 불일치 원인을 원본과 대조할 때 사용)
file_2022_original = next(DATA_DIR.glob("*2022*.xlsx"))
df_table1 = pd.read_excel(file_2022_original, sheet_name="Table 1", header=None)


def show_table1_around(center_excel_row: int, window: int, label: str):
    """Table 1 원본 시트에서 특정 엑셀 행(1-indexed) 주변을 보여줌"""
    start, end = center_excel_row - window, center_excel_row + window
    print(f"--- {label} (Table1 엑셀행 {start}~{end}) ---")
    # 1,3열은 병합셀로 생긴 빈 열이라 제외, 나머지는 알아보기 쉬운 이름으로 표시
    view = df_table1.iloc[start - 1 : end, [0, 2, 4]].copy()
    view.columns = ["세부사업명", "공통/자체", "예산"]
    display(view)


# TODO(2022): 아래 진단은 2024년 "충남 원본문서 결함" 케이스용 하드코딩이었음.
# 2022년은 §4 QA 불일치 목록을 먼저 확인한 뒤, 그 결과(어느 시도/중분류가 틀렸는지)에
# 맞춰 show_table1_around() 호출을 다시 작성할 것. 2024년 원본 진단 코드는 git main 이력 참고.

# 증감액 부호 소실 문제 확인

팀원 지적: 충남 "3. 모두의 역량이 고루 발휘되는 사회(공통)"는 2024년 예산(19,928)이 2023년 예산(42,867.5)보다 줄었으니 증감액·증감율이 음수여야 하는데, 원본에 양수로 찍혀있다. 이게 충남만의 문제인지, 다른 시도에도 있는지 확인한다.

방법: 세부사업별로 `2024년 예산 - 2023년 예산`을 직접 계산해서, 그 결과가 음수(예산 감소)인 행들만 골라 `증감액` 컬럼이 실제로 음수로 찍혀있는지 대조한다.

In [ ]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, '-'는 0으로 처리 후 숫자로 변환 (df_raw의 2022년 예산 변환과 동일한 규칙)"""
    return pd.to_numeric(series.astype(str).str.replace(",", "").replace("-", 0), errors="coerce")


# 이전 셀 실행 여부와 무관하게 독립적으로 동작하도록 2022년 예산도 여기서 다시 변환
df_raw["2022년 예산_num"] = to_numeric_budget(df_raw["2022년 예산"])
df_raw["2021년 예산_num"] = to_numeric_budget(df_raw["2021년 예산"])
df_raw["증감액_num"] = to_numeric_budget(df_raw["증감액"])
df_raw["계산된_증감액"] = df_raw["2022년 예산_num"] - df_raw["2021년 예산_num"]

# 실제로 예산이 감소한 행(계산된 증감액 < 0)만 골라서, 증감액 컬럼 부호를 대조
감소행 = df_raw[df_raw["계산된_증감액"] < -0.5].copy()
감소행["부호_소실"] = 감소행["증감액_num"] > 0.5

print("예산 감소 행 수:", len(감소행))
print("그중 증감액이 양수로 찍힌(부호 소실) 행 수:", 감소행["부호_소실"].sum())

In [ ]:
# 지역별로 부호 소실 비율 확인
지역별_부호소실_비율 = (
    감소행.groupby("지역")["부호_소실"].mean().mul(100).round(1).sort_values(ascending=False)
)
지역별_부호소실_비율

# 증감액 / 비율 직접 재계산
- 위에서 확인했듯이 원본에서는 시도별로 부호나 값 자체가 틀린 경우가 있어 신뢰할 수 없음. 
- 따라서 직접 재계산

In [ ]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, '-'는 0으로 처리 후 숫자로 변환 (df_raw의 2022년 예산 변환과 동일한 규칙)"""
    return pd.to_numeric(series.astype(str).str.replace(",", "").replace("-", 0), errors="coerce")


df_leaf["2021년_예산_num"] = to_numeric_budget(df_leaf["2021년 예산"])
df_leaf["2022년_예산_num"] = to_numeric_budget(df_leaf["2022년 예산"])
df_leaf["증감액_재계산"] = df_leaf["2022년_예산_num"] - df_leaf["2021년_예산_num"]
df_leaf["증감율_재계산"] = (
    df_leaf["증감액_재계산"] / df_leaf["2021년_예산_num"].replace(0, np.nan) * 100
).round(1)

df_leaf[
    ["세부사업명", "2022년_예산_num", "2021년_예산_num", "증감액_재계산", "증감율_재계산"]
].head(10)

# 최종 스키마

In [ ]:
# 텍스트 정리
PUA_PATTERN = re.compile(r"[\uE000-\uF8FF]")


def clean_text(series: pd.Series) -> pd.Series:
    """줄바꿈, 공백을 단일 공백으로 정리"""

    def _clean(x):
        if pd.isna(x):
            return x
        x = PUA_PATTERN.sub("", str(x))
        return re.sub(r"\s+", " ", x).strip()

    return series.apply(_clean)


df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"])
df_leaf["대분류"] = clean_text(df_leaf["대분류"])
df_leaf["중분류"] = clean_text(df_leaf["중분류"])

df_leaf.head()

In [ ]:
# 컬럼명 정리 + 연도 추가 + 최종 스키마만 선택
df_leaf_final = (
    df_leaf.drop(columns=["증감액", "비율"])
    .rename(
        columns={
            "2022년_예산_num": "당해예산",
            "2021년_예산_num": "전년도예산",
            "증감액_재계산": "증감액",
            "증감율_재계산": "증감율",
        }
    )
    .assign(연도=2022)
)

df_leaf_final = df_leaf_final[
    [
        "연도",
        "지역",
        "대분류",
        "중분류",
        "사업분류재정구분",
        "세부사업명",
        "주요내용",
        "당해예산",
        "전년도예산",
        "증감액",
        "증감율",
        "원본행",
    ]
]

print(df_leaf_final.shape)
display(df_leaf_final.head())

In [ ]:
year = 2022

for sido in df_leaf_final["지역"].unique():
    sido_leaf = df_leaf_final[df_leaf_final["지역"] == sido]
    sido_labeled = df_labeled[df_labeled["지역"] == sido]

    sido_dir = INTERIM_DIR / sido
    sido_dir.mkdir(parents=True, exist_ok=True)

    sido_leaf.to_csv(
        sido_dir / f"{year}_{sido}_세부사업_정제.csv", index=False, encoding="utf-8-sig"
    )
    sido_labeled.to_csv(
        sido_dir / f"{year}_{sido}_필터링전_전체원본.csv", index=False, encoding="utf-8-sig"
    )

# QA 결과는 전체 17개 시도 한 파일로 저장 (불일치 포함, 충남 결함도 그대로 남김)
qa.to_csv(REPORTS_DIR / f"{year}_전국_QA_검증결과.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ", df_leaf_final["지역"].nunique(), "개 시도")

# 세부사업명 오탈자 / 중복 탐지
- 오탈자를 자동으로 고치지 않고, 사람이 검토할 후보만 찾는다. 
- 같은 지역 / 중분류 안에서 완전히 같지는 않지만 유사도가 높은 세부사업명 쌍을 `rapidfuzz`로 찾는다.

In [ ]:
from itertools import combinations
from rapidfuzz import fuzz

In [ ]:
def normalize_for_match(name: str) -> str:
    """clean_text로 정제된 세부사업명에서, 비교 목적으로 공백, 문장부호 마저 제거"""
    return re.sub(r"[\s,-./\-/()]", "", name)


def find_near_duplicates(df: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    """같은 지역 중분류 안에서 완전 일치는 아니지만 유사도가 높은 세부사업명과 쌍을 찾는다."""
    candidates = []
    for (sido, midium_cat), group in df.groupby(["지역", "중분류"]):
        rows = list(
            group[["세부사업명", "당해예산", "주요내용"]].itertuples(index=False, name=None)
        )
        for (name_a, budget_a, content_a), (name_b, budget_b, content_b) in combinations(rows, 2):
            if name_a == name_b:
                continue
            if normalize_for_match(name_a) == normalize_for_match(name_b):
                continue  # 공백/문장부호 차이 -> 검수 대상 x
            score = fuzz.ratio(name_a, name_b)
            if score >= threshold:
                candidates.append(
                    {
                        "지역": sido,
                        "중분류": midium_cat,
                        "세부사업명1": name_a,
                        "당해예산1": budget_a,
                        "주요내용1": content_a,
                        "세부사업명2": name_b,
                        "당해예산2": budget_b,
                        "주요내용2": content_b,
                        "유사도": score,
                    }
                )
    return pd.DataFrame(candidates).sort_values("유사도", ascending=False)


near_dup = find_near_duplicates(df_leaf_final)
print(len(near_dup), "건 발견")
display(near_dup.head(10))

In [ ]:
# 당해예산이 완전히 같은 쌍은 진짜 같은 사업일 가능성이 높은 후보
near_dup_same_budget = near_dup[near_dup["당해예산1"] == near_dup["당해예산2"]]

print(len(near_dup_same_budget), "건 (전체", len(near_dup), "건 중)")
display(near_dup_same_budget)

In [ ]:
# 당해예산 0.0이 우연일치인지 확인
print("당해예산 = 0 으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] == 0).sum())
print("0이 아닌 값으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] != 0).sum())

near_dup_confidenet = near_dup_same_budget[near_dup_same_budget["당해예산1"] != 0]
display(near_dup_confidenet)

# 주요내용 구조 패턴 검사 
- 원자화를 위해 '지원대상:..', '지원내용: ..' 이 모든 행에 포함되어있는지 확인한다. 

In [ ]:
def dedup_label(text: str) -> str:
    """지원대상 : 지원대상 : ...  처럼 같은 라벨이 연속으로 중복된 걸 하나로 정리"""
    if pd.isna(text):
        return text
    text = re.sub(
        r"\(\s*(지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\s*\)",
        r"\1 : ",
        text,
    )
    text = re.sub(r"(지원대상\s*[:：]\s*)+", "지원대상 : ", text)
    text = re.sub(r"(지원내용\s*[:：]\s*)+", "지원내용 : ", text)
    text = re.sub(r"(사업대상\s*[:：]\s*)+", "사업대상 : ", text)
    text = re.sub(r"(사업내용\s*[:：]\s*)+", "사업내용 : ", text)
    return text


df_leaf_final["주요내용"] = df_leaf_final["주요내용"].apply(dedup_label)

In [ ]:
support_pattern = re.compile(r"^지원대상\s*[:：]\s*(.*?)\s*지원내용\s*[:：]\s*(.*)$", re.DOTALL)


def check_pattern(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern.match(text) else "불일치"


df_leaf_final["주요내용_패턴"] = df_leaf_final["주요내용"].apply(check_pattern)

print(df_leaf_final["주요내용_패턴"].value_counts())
print(df_leaf_final["주요내용_패턴"].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# 범위 넓혀보기
support_pattern_broad = re.compile(
    r"^(지원대상|사업대상|대상)\s*[:：]?\s*(.*?)\s*(지원내용|사업내용|내용)\s*[:：]?\s*(.*)$",
    re.DOTALL,
)


def check_pattern_broad(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern_broad.match(text) else "불일치"


df_leaf_final["주요내용_패턴_확장"] = df_leaf_final["주요내용"].apply(check_pattern_broad)

print(df_leaf_final["주요내용_패턴_확장"].value_counts())
print(df_leaf_final["주요내용_패턴_확장"].value_counts(normalize=True).mul(100).round(1))

# 여전히 안걸리는 샘플 확인
display(
    df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"][
        ["지역", "세부사업명", "주요내용"]
    ].head(20)
)

In [ ]:
def extract_via_regex(text: str) -> dict:
    """패턴에 걸리면 그대로 분리, 안 걸리면 None"""
    if pd.isna(text):
        return {"지원대상": None, "지원내용": None}
    m = support_pattern_broad.match(text)
    if m:
        return {"지원대상": m.group(2).strip(), "지원내용": m.group(4).strip()}
    return {"지원대상": None, "지원내용": None}


regex_extracted = pd.json_normalize(df_leaf_final["주요내용"].apply(extract_via_regex))
df_leaf_final["지원대상"] = regex_extracted["지원대상"]
df_leaf_final["지원내용_상세"] = regex_extracted["지원내용"]

In [ ]:
df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"].head()

# 주요내용 LLM 정제 (업스테이지 Solar Pro 3, 구조화 출력)

- LLM은 `주요내용_정제`(오탈자·이상한 공백만 보존형으로 교정) 생성만 담당한다. 요약/재구성 없음, 숫자·고유명사 변경 없음.
- `지원대상`/`지원내용_상세`는 이미 검증된 정규식(`extract_via_regex`) 결과를 그대로 쓴다 — 정규식으로 걸러지는 데 LLM을 또 쓰는 건 실익이 없고 과잉분리 위험만 늘린다.
- `response_format`(json_schema)으로 API 단에서 출력 구조를 강제, 실패 시 1회 재시도 후에도 실패하면 원문을 그대로 사용(정제문장 결측 방지).
- 정제 전후 숫자(금액·비율·연령 등) 보존 여부를 자동 검증해, 달라진 행은 검토 대상으로 표시한다.

In [ ]:
import os
import json
import yaml
from openai import OpenAI
from tqdm import tqdm

tqdm.pandas()

with open("../configs/llm_extraction.yaml") as f:
    llm_cfg = yaml.safe_load(f)

client = OpenAI(
    api_key=os.environ["UPSTAGE_API_KEY"],
    base_url=llm_cfg["upstage"]["base_url"],
)

RESPONSE_FORMAT = {"type": "json_schema", "json_schema": llm_cfg["response_schema"]}


def call_llm_once(name: str, content: str) -> str | None:
    """API 1회 호출. 파싱 실패하면 None 반환"""
    prompt = llm_cfg["prompt"]["template"].format(name=name, content=content)
    response = client.chat.completions.create(
        model=llm_cfg["upstage"]["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=llm_cfg["upstage"]["temperature"],
        response_format=RESPONSE_FORMAT,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)["정제문장"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


def clean_sentence(name: str, content: str) -> str:
    """주요내용을 LLM으로 정제. 결측이면 결측 유지, 실패 시(1회 재시도 포함) 원문 그대로 사용"""
    if pd.isna(content):
        return None
    for attempt in range(2):  # 최초 시도 + 1회 재시도
        try:
            result = call_llm_once(name, content)
        except Exception as e:
            print(f"API 호출 실패({attempt + 1}회차): {name} -> {e}")
            result = None
        if result is not None:
            return result
    print(f"정제 실패, 원문 유지: {name}")
    return content


def extract_numbers(text) -> list:
    """숫자(금액·비율·연령 등) 시퀀스만 뽑아 리스트로 반환 - 원문/정제문장 보존 검증용"""
    if pd.isna(text):
        return []
    return re.findall(r"\d+", text)


def numbers_preserved(original, cleaned) -> bool:
    """정제 과정에서 숫자가 그대로 보존됐는지 확인 (다르면 검토 대상)"""
    return extract_numbers(original) == extract_numbers(cleaned)

# 먼저 소량 샘플로 속도·품질 확인 (전체 실행 전 확인용)

```python
sample = df_leaf_final.sample(
    llm_cfg["sample"]["size"], random_state=llm_cfg["sample"]["random_state"]
).copy()

sample["주요내용_정제"] = sample.progress_apply(
    lambda row: clean_sentence(row["세부사업명"], row["주요내용"]), axis=1
)
sample["숫자보존"] = sample.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~sample["숫자보존"]).sum())
display(
    sample[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세", "숫자보존"]]
)
```

샘플 결과가 괜찮으면 아래 셀로 전체 실행 (7천여 건, 시간 오래 걸릴 수 있음)

In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / "2022_llm_정제_체크포인트.csv"

PUA_LOW, PUA_HIGH = chr(0xE000), chr(0xF8FF)
pua_re = re.compile(f"[{PUA_LOW}-{PUA_HIGH}]")
paren_label_re = re.compile(
    r"\((지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\)"
)


def needs_llm_rerun(text):
    if pd.isna(text):
        return False
    return bool(pua_re.search(text)) or bool(paren_label_re.search(text))


# 체크포인트 파일을 직접 읽어서 판별한다(df_leaf_final 병합을 기다릴 필요 없음).
# 이렇게 해야 LLM 처리 셀보다 앞에 둘 수 있고, 한 번의 순차 실행(Run All)만으로
# 재실행 대상까지 자동으로 포함되어 처리된다 (CodeRabbit 리뷰 반영).
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    affected_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
    affected_idx = checkpoint_df.index[affected_mask]

    if len(affected_idx) > 0:
        checkpoint_df = checkpoint_df.drop(index=affected_idx)
        checkpoint_df.to_csv(CHECKPOINT_PATH)

    print(
        "재실행 대상(체크포인트에서 제거):",
        len(affected_idx),
        "건 -> 남은 행수:",
        len(checkpoint_df),
    )
else:
    print("체크포인트 파일 없음 - 전체 신규 실행")

In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / "2022_llm_정제_체크포인트.csv"
CHUNK_SIZE = 200

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    done_index = set(checkpoint_df.index)
    print(f"체크포인트 발견: {len(done_index)}건 이미 처리됨, 이어서 진행")
else:
    checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])
    done_index = set()

targets = df_leaf_final.index.difference(done_index)
results = {}

for i, idx in enumerate(tqdm(targets), start=1):
    row = df_leaf_final.loc[idx]
    results[idx] = clean_sentence(row["세부사업명"], row["주요내용"])

    if i % CHUNK_SIZE == 0:
        partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df.to_csv(CHECKPOINT_PATH)
        results = {}

# 남은 것 마저 저장
if results:
    partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
    checkpoint_df = pd.concat([checkpoint_df, partial])
    checkpoint_df.to_csv(CHECKPOINT_PATH)

df_leaf_final["주요내용_정제"] = checkpoint_df["주요내용_정제"]
df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~df_leaf_final["숫자보존"]).sum())
display(
    df_leaf_final[~df_leaf_final["숫자보존"]][
        ["지역", "세부사업명", "주요내용", "주요내용_정제"]
    ].head(20)
)

df_leaf_final[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세"]].head(20)

In [ ]:
output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]

missing_cols = [c for c in output_cols if c not in df_leaf_final.columns]

if missing_cols:
    raise KeyError(f"df_leaf_final에 없는 칼럼: {missing_cols}")

for sido_name, group in df_leaf_final.groupby("지역"):
    out_path = INTERIM_DIR / sido_name / f"2022_{sido_name}_세부사업_정제.csv"
    group[output_cols].to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")


print("\n숫자보존 불일치 총 건수:", (~df_leaf_final["숫자보존"]).sum())

In [ ]:
df_leaf_final.columns.tolist()

In [ ]:
id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]

df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

# 전년도예산 행은 실제 그 돈이 집행된 연도로 맞춰서 -1
previous_mask = df_long["예산구분"] == "전년도예산"
df_long.loc[previous_mask, "연도"] -= 1
# 증감액/증감율은 "당해 대비 전년" 개념이라 전년도 행에는 그대로 복제되면 안 됨 (CodeRabbit 지적)
df_long.loc[previous_mask, ["증감액", "증감율"]] = None

df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf_final), "행 x 2)")
df_long.head(10)

In [ ]:
# 시도별로 저장
for sido_name, group in df_long.groupby("지역"):
    out_path = INTERIM_DIR / sido_name / f"2022_{sido_name}_세부사업_정제_long.csv"
    group.to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")